> Open from the repository root or set `HATEWIC_CODE_ROOT`. Private prepared artifacts and fresh validation gates are required before training; see `docs/reproduction.md`. Saved thesis runs are not included.

# HateWiC Local Apple GPU Pilot
This independent notebook uses the existing Python environment and the Apple M5 GPU. It preserves the Colab source, original examples, labels, model settings and frozen folds. Outputs are local and private.

This is a training/validation execution pilot, not a formal test result. MPS uses float32; the earlier CUDA runs used float16 autocast. Do not silently combine backend results or continue a CUDA recovery state on MPS.

## 1. Verify the environment

In [ ]:
from pathlib import Path
import os,sys,json
ROOT=Path(os.environ.get('HATEWIC_CODE_ROOT',Path.cwd() if (Path.cwd()/'src').is_dir() else Path.cwd().parent)).resolve()
os.environ['HF_HUB_OFFLINE']='1'
os.environ.pop('PYTORCH_ENABLE_MPS_FALLBACK',None)
sys.path.insert(0,str(ROOT/'src'))
import torch
import hatewic_formal.training as training
assert ROOT in Path(training.__file__).parents
assert torch.backends.mps.is_available(), 'Apple GPU unavailable; stop rather than use CPU.'
print('Python:',sys.version.split()[0], 'PyTorch:',torch.__version__)
print('GPU: Apple M5 via MPS; precision: float32')
print('Working directory:',ROOT)


## 2. Inspect recovery and implementation checks
The hardware check covers all seven variants. Synthetic recovery is numerically checked at absolute tolerance 1e-7; bitwise identity is reported separately.

In [ ]:
import subprocess
checked=subprocess.run([sys.executable,'-m','pytest',str(ROOT/'tests'),'-q','--ignore='+str(ROOT/'tests/test_mps_gpu.py')],cwd=ROOT,env={**os.environ,'PYTHONPATH':str(ROOT/'src')},capture_output=True,text=True)
print(checked.stdout)
if checked.returncode:raise RuntimeError(checked.stderr or checked.stdout)
print(json.loads((ROOT/'verification/mps_recovery.json').read_text()))


## 3. Run or reuse the local GPU pilot
Each real model uses 64 training and 32 validation examples from rotation 0. No test IDs enter training, validation or timing. Actual batch 8, accumulation 2, context/gloss limits 128/64 and the losses are unchanged. Each pilot deliberately interrupts after one optimizer update, then reloads recovery state. A completed verified pilot is reused.

In [ ]:
sys.path.insert(0,str(ROOT))
from local_pilot import main
pilot_status=main()


## 4. Review measured results

In [ ]:
import pandas as pd
report=json.loads((ROOT/'verification/local_pilot.json').read_text())
assert report['status']=='passed'
rows=[]
for item in report['benchmarks']:
    rows.append({'model':item['model'],'pilot_seconds':item['elapsed_including_resume_and_validation_seconds'],
      'sampled_MPS_tensor_GiB':item['result']['sampled_mps_allocated_bytes_max']/1024**3,
      'MPS_driver_GiB_after':item['mps_driver_allocated_bytes_after']/1024**3,
      'precision':item['result']['precision']})
display(pd.DataFrame(rows))
print('Both local pilots passed. Formal batch training has not been started by this notebook.')
print('Memory figures are sampled allocations, not a continuously measured peak or total system RAM.')
